# Analisi dell'errore sulla predizione di immagini

In [93]:
# COMMON LIBRARIES
import os
import cv2
import requests
import json
import numpy as np
import torch
import random

import matplotlib.pyplot as plt
from datetime import datetime

# DATA SET PREPARATION AND LOADING
from detectron2.data.datasets import register_coco_instances
from detectron2.data import DatasetCatalog, MetadataCatalog

# VISUALIZATION
from detectron2.utils.visualizer import Visualizer
from detectron2.utils.visualizer import ColorMode

# CONFIGURATION
from detectron2 import model_zoo
from detectron2.config import get_cfg

# EVALUATION
from detectron2.engine import DefaultPredictor

# TRAINING
from detectron2.engine import DefaultTrainer

Dati globali

In [94]:
# Percorsi ai file COCO
base_path = os.getcwd()

# Costruisci i percorsi relativi alla posizione dello script
json_file = os.path.join(base_path, "../../annotations", "instances_val2017.json")
image_folder = os.path.join(base_path, "../../val2017/")

# Nome dell'immagine di cui trovare la bounding box
image_filenames = ["000000409475.jpg", "000000000872.jpg", "000000002153.jpg"]  # Sostituisci con il nome dell'immagine desiderata

#Modello pre-addestrato
MODEL_USED = "COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"

### Estrapolazione dei dati
Estraggo dal json delle annotazioni le informazioni relative alle bounding boxe delle immagini da predire

dati: [x_min, y_min, width, height]

In [95]:
def extract_pixels_from_coco(image_filename, json_file):
    # 1. Caricare il file JSON con le annotazioni
    with open(json_file, "r") as f:
        coco_data = json.load(f)

    # 2. Creare una mappatura tra category_id e category_name
    category_dict = {category["id"]: category["name"] for category in coco_data["categories"]}

    # 3. Trovare i dati dell'immagine corrispondente al nome file
    image_data = next((img for img in coco_data["images"] if img["file_name"] == image_filename), None)

    if image_data is None:
        print("Immagine non trovata nel dataset COCO!")
        return []

    image_id = image_data["id"]
    image_width = image_data["width"]
    image_height = image_data["height"]

    # 4. Estrarre le segmentazioni associate a questa immagine
    extracted_data = []
    for ann in coco_data["annotations"]:
        if ann["image_id"] == image_id:
            category_id = ann["category_id"]
            category_name = category_dict.get(category_id, "Unknown")

            # Converte le segmentazioni in array di interi (ogni istanza può avere più poligoni)
            segmentation_pixels = [np.array(seg, dtype=np.int32).reshape((-1, 2)).tolist() for seg in ann["segmentation"]]

            extracted_data.append({
                "category_name": category_name,
                "pixels": segmentation_pixels,
                "image_width": image_width,
                "image_height": image_height
            })

    return extracted_data

def visualize_masks_on_image(image_filename, instances, image_folder):
    # Caricare l'immagine originale
    image_path = f"{image_folder}/{image_filename}"  # Percorso completo
    image = cv2.imread(image_path)  # Caricare immagine
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)  # Convertire in RGB per Matplotlib
    
    if not instances:
        print("Nessuna istanza trovata per questa immagine!")
        return
    
    image_width = instances[0]["image_width"]
    image_height = instances[0]["image_height"]

    # Creare una maschera vuota a colori
    mask_overlay = np.zeros((image_height, image_width, 3), dtype=np.uint8)

    for instance in instances:
        category_name = instance["category_name"]

        # Scegliere un colore casuale per questa istanza
        color = [random.randint(100, 255) for _ in range(3)]

        # Disegnare i poligoni sulla maschera
        for poly in instance["pixels"]:
            polygon = np.array(poly, dtype=np.int32)
            cv2.fillPoly(mask_overlay, [polygon], color=color)

    # Sovrapporre la maschera all'immagine originale con trasparenza
    alpha = 0.5  # Regola la trasparenza
    blended = cv2.addWeighted(image, 1, mask_overlay, alpha, 0)

    # Mostrare l'immagine originale e quella con maschere
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    ax[0].imshow(image)
    ax[0].set_title("Immagine Originale")
    ax[0].axis("off")

    ax[1].imshow(blended)
    ax[1].set_title("Segmentazione Sovrapposta")
    ax[1].axis("off")

    plt.show()


In [ ]:
gt_instances = np.array([])
for file_name in image_filenames:
    # Estrarre i dati delle istanze segmentate
    instances = extract_pixels_from_coco(file_name, json_file)
    gt_instances = np.append(gt_instances, instances)
    # Visualizzare la segmentazione
    visualize_masks_on_image(file_name, instances, image_folder)

print(gt_instances)

In [97]:
def extract_polygons_from_masks(masks):
    """
    Converte le maschere binarie in poligoni (segmentazioni precise).
    """
    polygons = []
    for mask in masks:
        # Trova i contorni della maschera binaria (maschera di 1s e 0s)
        contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for contour in contours:
            # Appiattire il contorno in una lista di punti (x, y)
            polygon = contour.reshape(-1, 2).tolist()
            polygons.append(polygon)
    return polygons

def process_detectron2_output(image_filename, image_folder, model_used):
    # 1. Caricare l'immagine
    image = cv2.imread(image_folder + image_filename)
    
    # 2. Configurare il modello pre-addestrato di Detectron2
    cfg = get_cfg()
    cfg.merge_from_file(model_zoo.get_config_file(model_used))
    cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5  # Imposta la soglia di confidenza
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(model_used)
    cfg.MODEL.DEVICE = "cpu"  # Usa la CPU (puoi cambiare con 'cuda' se hai una GPU)
    
    # 3. Inizializzare il predittore
    predictor = DefaultPredictor(cfg)
    
    # 4. Eseguire il modello sull'immagine
    outputs = predictor(image)
    
    # 5. Estrarre le istanze predette (classi, bounding boxes)
    instances = outputs["instances"].to("cpu")
    
    # 6. Ottenere il metadata (nomi delle classi)
    metadata = MetadataCatalog.get(cfg.DATASETS.TRAIN[0])
    
    # 7. Estrazione delle informazioni delle classi, bounding boxes e maschere
    category_ids = instances.pred_classes.numpy()  # ID delle classi
    bboxes = instances.pred_boxes.tensor.numpy()  # Bounding boxes
    masks = instances.pred_masks.numpy()  # Maschere binarie per ogni istanza
    
    # 8. Creare un array delle istanze predette con le informazioni desiderate
    boxes_predected = []
    
    for category_id, bbox, mask in zip(category_ids, bboxes, masks):
        category_name = metadata.thing_classes[category_id]  # Ottieni il nome della categoria
        segmentation_pixels = extract_polygons_from_masks([mask])  # Poligoni precisi
        
        # Aggiungi l'istanza al risultato
        boxes_predected.append({
            "category_name": category_name,
            "pixels": segmentation_pixels,
            "image_width": image.shape[1],
            "image_height": image.shape[0]
        })

    # 9. Mostrare il risultato: maschere sovrapposte sull'immagine
    visualizer = Visualizer(image[:, :, ::-1], MetadataCatalog.get(cfg.DATASETS.TRAIN[0]), scale=1.2)
    out = visualizer.draw_instance_predictions(instances)
    
    # Convertire l'immagine da BGR a RGB
    image_result = out.get_image()[:, :, ::-1]
    
    # Mostrare l'immagine con Matplotlib
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(image[:, :, ::-1])
    plt.title("Immagine Originale")
    plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(image_result)
    plt.title("Immagine con Maschere")
    plt.axis("off")
    
    plt.show()

    # Stampare le istanze predette
    for entry in boxes_predected:
        print(entry)
    return boxes_predected

### Predizione dei dati con Detectron2

In [ ]:
boxes_predected = np.array([])
for image_filename in image_filenames:
    prediction = process_detectron2_output(image_filename, image_folder, MODEL_USED)
    boxes_predected = np.append(boxes_predected, prediction)


### Calcolo del root min square error

$$RMSE = \sqrt{\frac{1}{n} \sum_{i=1}^{n} \left( (x_i^{\text{pred}} - x_i^{\text{gt}})^2 + (y_i^{\text{pred}} - y_i^{\text{gt}})^2 \right)}




1.	Confrontare maschere binarie: Per ogni istanza, devi confrontare la maschera predetta con quella di verità a terra.
2.	Calcolare la differenza: Sottrarre la maschera predetta dalla maschera di verità a terra.
3.	Calcolare il quadrato delle differenze: Elevare al quadrato la differenza di ogni pixel.
4.	Media degli errori: Calcolare la media degli errori quadratici.
5.	Radice quadrata della media: Calcolare la radice quadrata per ottenere il RMSE.


In [99]:
# Funzione per calcolare l'errore quadratico medio (RMSE)

def calculate_rmse(predicted_instances, gt_instances, min_overlap=0.0):
    """
    Calcola l'errore RMSE tra le segmentazioni predette e le segmentazioni di verità a terra per ogni istanza,
    considerando solo le istanze della stessa categoria e con una sovrapposizione sufficiente tra le maschere.
    
    :param predicted_instances: Array con le segmentazioni predette
    :param gt_instances: Array con le segmentazioni di verità a terra
    :param min_overlap: La percentuale minima di sovrapposizione tra le istanze (default 0.5)
    
    :return: Array con gli errori RMSE per ogni istanza e la media degli errori
    """
    
    errors = []
    
    # Itera attraverso le istanze predette e le GT
    for predicted, gt in zip(predicted_instances, gt_instances):
        # Verifica che le categorie siano le stesse
        if predicted['category_name'] != gt['category_name']:
            continue  # Se le categorie non sono uguali, salta l'istanza
        
        # Estrai le segmentazioni binarie per la predizione e la GT (in formato maschera)
        pred_mask = np.array(predicted["pixels"], dtype=np.uint8)
        gt_mask = np.array(gt["pixels"], dtype=np.uint8)
        
        # Calcolare la sovrapposizione tra le maschere (intersezione/union)
        intersection = np.sum(np.logical_and(pred_mask, gt_mask))
        union = np.sum(np.logical_or(pred_mask, gt_mask))
        overlap = intersection / float(union) if union > 0 else 0
        
        # Se la sovrapposizione è inferiore alla soglia minima, ignora questa coppia
        if overlap <= min_overlap:
            continue
        
        # Calcolare il RMSE (radice quadrata della media dei quadrati delle differenze)
        
        # Calcola le differenze quadrate tra i dati
        differences = pred_mask - gt_mask
        squared_diff = differences**2

        # Somma delle differenze quadrate
        sum_squared_diff = np.sum(squared_diff)

        # Calcola RMSE
        rmse = np.sqrt(sum_squared_diff / len(pred_mask))
        errors.append(rmse)
    
    # Calcolare la media degli errori RMSE
    if errors:
        mean_rmse = np.mean(errors)
    else:
        mean_rmse = 0  # Nessun errore se non ci sono istanze valide

    return errors, mean_rmse

## Calcolo degli errori

In [ ]:
# Funzione per associare le predicted boxes alle GT boxes e calcolare gli errori

print("PREDIZIONI\n",boxes_predected)
print("GT:\n",gt_instances)
